# UEL_SGS_LucViHoangDe_C1

**Chosen approved backbone:** `distilbert/distilbert-base-multilingual-cased`

**Team:** Lục Vị Hoàng Đế — UEL / SGS. Primary configuration: `configs/distilbert_ce.json`.

Fine-tune one approved encoder, evaluate macro F1 on a group-aware split, then retrain on all training rows. Attach the competition and the [approved Kaggle model](https://www.kaggle.com/models/aisuko/distilbert-base-multilingual-cased/transformers/default) in Input and enable a GPU. Source code is embedded; competition IDs, labels, prompts, predictions and split data are not embedded. No GitHub checkout or external pretrained backbone is fetched. This notebook creates a CSV and never submits to Kaggle.

Prepare compatible dependencies before an offline run: PyTorch >=2.8,<3, Transformers >=4.56,<5, NumPy >=2.1,<3, pandas >=2.2,<3, scikit-learn >=1.6,<2, safetensors >=0.5. If packages are missing, install compatible versions from PyPI while internet is enabled and restart the kernel. Keep a compatible CUDA PyTorch build.

Overview/Data were supplied by the participant. Live Rules could not be independently verified due Kaggle rate limits. Check current Rules/Discussion before submission. No external data, translation model, pseudo-labeling, or multi-backbone ensemble is used.

In [ ]:
import os, sys, json, base64, zlib
from pathlib import Path
from packaging.version import Version
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
MODEL_ID = 'distilbert/distilbert-base-multilingual-cased'
CONFIG_PATHS = ['configs/distilbert_ce.json']
MODEL_DIR = None  # Set explicitly if automatic discovery is ambiguous.
DATA_DIR = None
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
os.chdir(WORK)
import torch, transformers, numpy, pandas, sklearn
for name, current, lo, hi in [('torch',torch.__version__,'2.8','3'), ('transformers',transformers.__version__,'4.56','5'), ('numpy',numpy.__version__,'2.1','3'), ('pandas',pandas.__version__,'2.2','3'), ('sklearn',sklearn.__version__,'1.6','2')]:
    assert Version(lo) <= Version(current) < Version(hi), f'Prepare compatible {name}; found {current}'
assert torch.cuda.is_available(), 'Enable GPU for a full training run.'
print({'torch':torch.__version__, 'transformers':transformers.__version__, 'GPU':torch.cuda.get_device_name(0)})

In [ ]:
# Source matches protector/ and configs/ in the repository.
payload = 'eNrtPWtz20aSfwWn1F6BNghRku1NlGXqHMfOeuM4LtvJfqBYWJAcirBAgAFAyVqd/vv1a14ASEneS91+uH1YBGamp2emp6efg5uDTVU2at6U1WGSZEXWJEm8uT44DQ7O6L/vFVRYbOfZLFdRkG7g6VIthrN0fjErCxXA83rTBPM8retsmc3TJiuLYFlWwfufX38MjkfHz2KGdFacFUlyqaoaaiRJMA7ODkbxUTzCooMocDCZp3k2qwiUIJOtN2XVBJ/qEqAsoc9gkzYrqBVIyTt4xB7ksdiuN9dBWgfFxrzbpMUC3sD/NgsBUl/kKq2KeK2aKpvXGtjyKKnnZaXatcqFypNa5YAljlJqN1WaFUmj6iapN3nWnGkU43lZ1E1aNAbwm+ffv3xz/PoHXWGRNqkuu6qyRiU8QvzvQi0DPREqVJ83qsrWqmgGp2dFAP/hWf24hUUAXFZpvgyalQouocmCV4Ggw6BhyLWqa6yGNUr4p6IGMfaDsD6uMijeYKs0hyZpfl3Dm/W2hqksm2CmAhjLclurRXCVNasgDQp1FWSA5EbBPwXMAoxfMJuv1PxiU2bw1k5WmlcqXVwHBAPRuFqVuYdurZrYHRr/tgMHisFVdqeCqyyrdK2gdLOIsY9kXl86lYJDgGe7STaVWmSEUx1DzbMDgZKX5xms1JjBTSbLswN6ldzk6Uzlt2cHRNb0ACM3azmdxk2ZEMGFAupaQ4mpdrxON6GuPujUbmAJI540aNemprDYxGmVFucqzFUREtjBIAq4RvZPNR7FT6MAaizKdQLk1qjx0cmfI4a9/z91g5tseT3WyBbn2/RcBY9hxhIY72N3FIItkiVvlzBbfI6CWZbWmiTxP5VqthVwgLxMm1BvpPB6ApWnURDyLNMTgKfGMLzzdfo5PIJRUU/1eDKKgqMoOIYWKXAMwGl8drBO51V5Bpzin6oqk0V2mSEnGY8Gghm+hgmE+cJfdXgi72c4U9hTxD+XR1ALq0R6ILwC+EpTFCz0J1zkcEizO4pH+M9Td6BYZ8N1jrDYrwn4xyO3OmECSDCGaVWl1+GEan+Kgs104NcEet0iSXsI0mz59bKlVP3OjO1xcKSGX5921793HviZYMhspbVKiBi5Q7VINGkKMlJI0xW1Xpo+BE/e/7Djx7YIkfYgfzd2egX8YVZGxwHQgaJOGFKl6m2OPODm7EBDJXBncD7oF7C38qxuQsDr7GAOWwI3vTLVDAq2Xnuazg5grrPiPCFqgzlKELU8KxQC6NJL1NPCdGz6XB71dMRcGVlUf2fuSvTX9jtyprSvt/lcbWCKqGpZ5sRXiutQT93AHUpVXtGEIcfBwQ5aGLjl1GHfRM5h66YNVjs7eG7a8kl1BWSwUvkiKLdw3OJZSJTB3QMZwVusArz6Miu3dS7nBuy2vsMlDl7A6ZRVazxLUUDBjui8S6ESnFyLgHjpYa0AyEzB1lVAUPk1dIa16nWa5wAZuWwdB2/LAA6PoWGHzQqID06rRbDMmobaVMR/hV0RbiDl3PIc2IO8fQq5cg0d9MDMmK5lu2wqGFeIRfECjog65NKITtqiGR/DQizzbb0af6y2aqB3BnFcrtoRpfLMF6GA2W7SqlZGLLpbpurKKul2AQcjvon0Ca6SejtbZzXy5KhXmFnDwRZqlkg4VLCdNT7x8+p8izP1jkrCharnVUYiCfB+I0xaEVQVc5DHgAdnG9kymudsZygMEJg4XSwQMX6qEWoD4Obleg17h+f/920GIoE7ozQ8gAENCQC3DmELYIHpiJ6oQiq4Q5XhEKdlL+RuE9gFm22DjWCiUlhGwDGt4GhO50196PdKAkIfclTgV3sYcjuasFC7D7dDquJJUztgoRSZnT8QAZK7h0icD2xop7Wv0TJDYbdnIqkAJCszGHrxsLnc0cQyhAc2vHsOdjTcT1rU6K5x8vFpWBSJz33zJkXtqg+buJ2NLON/cNP9dLCzmTfueT+tGA3Nrfgl667ZaF8nusz0oV88bGp3t7Ks++Ft9eRqDled15b90h+sX2ttB4Q/fIyFBQdjtAMIh3OEVnvChFSdzxr6yR0KOGD9fQCFF7riOh1i9B5PcE93d1bH1G+XdNAgnhfpvpGtyQObCXCrPgBhy3PuxNlUtfXkTQdHi/5+3OTBEdx3oGm2+J1Yaj4BUly1NVq3oyfalx207Vb/MiydTdnB05HBNKq2uqlrDS+teTQd18pTekU56RGGGIIrHJnhOsqcHgePtKW9ORKlNx2e7Hg/+XHA8hh0lyQF6PZoiMMpSxIUz5LEzBhLax1xUlu0RKgkq8YHGPjk7OD7l29f//gWGcjZwd+ev37z/fuXz3/ix3fvf/n53cfk9du/vXzx8fUvUGl6VmiLCOp0JEifBhlp1VlkrSyqAE5DC8FdDW6x4dsff33+40vpd9NwJ8uK/2byrArq5qz4KvilAA2iWakaDZaKSBUVATFi1jD1CkX+NYj3+J7qAhvewBaHZ5I7f7lU1WWmruKz4vk7GM9vLwlxbbBagDqZ5TNVNYf25xBVuOEaViADCRWUiXw4RyWNdaI0q7cX5T2qHwLCRQ0zAzNRH8oxDmPUfX9abYfzvN4crtffv3z/cUjqjNfHek3wqeCh0BCpPmD4/i5Yr9K5AmXz4vnrw8/5eliV2DLtgmwXfiHcHHbHbsBUehfk87I8z9VhpWiQHix5dxcE2SxpBmXc8eUJryzbyodZ8YnV1uHlsU8Jd9e/s/O0qcpiWw/rBg7Q8vCqzJdDqIZG2qoD0Ot9f9W7Os5m6+E5VAFGdSh/hyfxyfB4BjDqptrO/cncUeeuboCHQo0cJvdn/jX88/fD19J4eDmKT3xS1ZVsD1Rpdze3yDBevn3xyw8v3yN/qVUT6v0+CIZ2w99rwA/Cm3v3+S2eEr7+vkpr1NFb6rs8VFat3xYZ6sckD/4RHpN+X8gHtigD1/yxKrebn16V+eIuf8iHKDAc3VoLnDOPxZVy9skaD5qV9gbg74F9jRInnHTx+mKRVSE/1CTIRoH6DHOflBee8I9tuK9GffbOTugQ2hT1tgIJtp5n2fhVCse+PUzhJM/z8grO0YKLBlgfZh2YN2hZ22Y5/JrkYj2mepUeP33GKMtQcByypLEUC2bkZ7FDjMuNKkDgrmYAEpdk2bJFz/JyfoFHJoylCvN0PVukp8GSXCLh0ej4SfAowD+A5AzdK23j9CreblB0CQmQb1taxSv1eZGdK7SZ2vGQt4VkdJhsItjriBw2ZJxF5wRPve7K1NKLZ164nhxUH25urfdH7Mc3Fl0r3+P5ny34vMf141/acqefQJYAUSDyAMBQ7tHeb1Sn603uynYtCK2ebq0zAUWtKJiX+XZd1CTXyMBiWK11HXqOhR53lp25w2B5dnCD8G7Z4AKKfXO9UWPgJVFwodQmEX4GdClk6YmZ7NIRVIwhPPiPsUavRRcw0yA2/YYegpdVVVah6f/ULo8e2Y38uI1g2FuQx2/6u7s1+qKHVbaAvQdC1xzocBEOyEA9QGurKYZR4v+zDZSq30MiZK72ALRnsMAXCNZ0Frz+oR+jFGTAa9lNQX0a1L0YCAoPRkTBGXsN6psCLtnq3qVCJBimClRx0CGLx5LvtBsEf+HTyvDShyCyLS6K8qoIbJ9dbIi296OCbkKLB8nrX4QEdTVobYl6gvWm2reqLX/oCo0C3ppQFuq6U9547mazJgifSdFYyL3aVOiExB9KfjDgAW2Q8OjJaDSKgmfmX8/z2R7a2cFLvT+OnkRQW2y32NszfgYudCios0/FoohzCKSWLQbBf/IDboA7OvyIHRySbwIUqxJ0ljzddKASIBwQPnH39wDt4Pr6hxqY+XKpKgNbTgp/PexRUYCsRTaKBFYlOUfhIESGqzv9KvgVHTwlKmpLb2dSXThRvw1I5oD1BN23xv5wLkFRyyrYytk5WXMRZNxyi6DuHQKiZ2f1Y+bTAf5xBKTYYAfjfPvqpxdYTtjFqIAtQYgJYY/LrnfPc8ALDwSaVDuYq7Ja4CkGfS9BWAApgfu/ov53ToU/kzeEZ/ypBD2cAE6y0yx4HJxMB6wlI2VzYAB6z0dMrlQTBVVQ9W8tpmY6ub+ad0lkXVvjUfzNyA8tIQnOYruwQHjDAHGRpBf8nBV/BRFm+ObDXwPjkgzWaTMHjdvEmDxHx83nbI1lthZq45m6hKWDdwEeq7rhtyhdQTkKitCJvEVVXSySuvttjWYn9TmdN3pFgr+l83laLeLgDXvoCgXVQTgD0Sios3UGumAGnLcGWXa+6ok6YcERlpCOq074xUAPiqSgsmzCzN09V6sMcGAYk2yKOy1r8UFbONa/zatpy8dvq2TTTqBF5qICNI02IpDwXHSkMeGZDrBH+jkzo3DW2IpemzSr6kjPKJLzZBrB//V+PXoWzGAZ6+Bz8AQWDnSG9Nvg7Xb97hql+5x8sGxmK5SCakUJawQMMPgpPT+n6ZlfwEmjdyssIUVGcCBL/J7+fMBwlvDJsbYtI+rFOdVBQxeGiDx6dHIM5A4/KRjm2RMtDwGsLVR69kSHgbQajx7UeDu/UI0rmYqhCjeub6fis9BlB3qKcYC79r532lJlAGrXpUU9n1xQiwn+bBENUwIg+GnQJjxYVhRrQM8PqQKFqiBbtBAT2k0o2NjGLZOnTzeMAmCVORb08gK0J9Sh22zS1tHUpRHiNk6Fr4LfFEi9wKNQH11VwI5rZhXID9BUh277ufrWJWLmBfB+gYQLR8Q69uaXDnjuKfhL0I6VAS21yYqtYw1GBY32ho3cAeKIUalNZtfAD0OtwoFkeaGOZ2Eds3s6tGogkBYpUBy09WQQa32KNIesaXKUuoiuSEOoie9pPKe7KJNstbT7UPgJ08lpFLwtC5CTHmm8Hwcz83YQ/CkwIGBnffPkm2d/Pjk6guMNoIQpqMjjIwf0hbqmrR/iXo+kpwk+APwnp/QaQ44G+IiRcDwfAx4IlZpj6ujZYOqa2OUMqMXMMvD1WegZmyIC7QUyLY3KytszPgcw0CAKwsGgBe6TM6cWQFs8pW2DhIqReEy//22IdPKptcmck2TsEBXKa7bJAFQ2LDOguyFcDpzvxvZM7onh2rmtd21tCxrJTO/ET3w8+ht8/6TrKYYBiH7JMz2ZDnSXmQAjtubtFjki23KLc6JOB/bUSRwN+AcQz17hYyjnkWiSY1C7QWJOUpbo8OeMf9oR8/NaNasShjq1QjD2azvyj0r9ltR6BM8ntH4dU6cU0al10sHu5rPe5rO7m/uNFlUJWr2RvkKDG7k7Fnrk7ZGKhEDLETnwrFjoeFkdC474K099DSshgbZj8JG+uI2263geT364t0WOBVTGF1lDR3QFHU087VVZ1y6xAOeGiuGkqSYdu850isPoew8F2TmcHipBw95nF5eETHzJTlQcDDycdAPTeBKan0xBuNFF4UT9rlU8g2NJl041xZoqA46NNM+OvIbBxWgvkuXAoLMCZFyHeBKOlRHTEY/YtRI5A7gvMIpW3g2RHOUYPgpHo8Q0I03pdcaYvLER8TiSqNuxE2HU2wcIinXbSKijFG8ce6FR8q0JUNT9W8/Ut8xQYCGDLIFA08OpNuC2bJeAIBYPjKWPooKlU40z92efOrZECZ/y0eAAQ8SgqSSYnGJ1k3m5hW0UUjQ5+dAHUcc+mmhrjgXAz/eEgcKKD0I9EARhDCJSYu1Kp7RRkciadBY6SEVmiDuhqeK8WSUg8FUg36mKF9efMBL9HLIkq35iDCG82mJaIVnUGldYB/egORKxA3I3NGLsPTL+oNXN7nqt/r3tJjjwnpfOmSHsa+RsaBOya1/5TQkJXlSsi2qSHiFv1zaCtht9+KD37a0VxdGacqiNKIegGMMMguYH8vpCNehxL9D9PQ+ePRmSbLlda3NCgOaE0NUy0b1Ckj0aOYZNlZ0D49WqPjLU+BvQY74bf23Er285XFSsDqztAx+69gwNsfHTiZUfWY1QNgWXE+vZYNAcRe7eQfkdqz8fUMgSbJAmEjsMJ9RcCM+mgSuIMK2bdBSmGjQ943q4ChQyvknv3pgae+kNNAx/H5xKVobAjn/fpqDp5AqKmHf9TljCHGL+QvwN/p9+wK+jQTe62WHPKBMkHNyqA5sJM51bkxVsQTg7+Cr4gdKSOOIrIqsL/vuC5pMMS6S2cFDGvFxDF0BH5EDMS4zCw3eK1yJANi0R25yIxvp4HWy2szxD7Sd2O/nHP/6h0XMcfYTpflffQFq7wH5zUpfoiNuibtrnAcUYcxNNfrYdjRZ/ZiN3ZKybtB3IVIzb1zG1xcEL3K0chSuHJk0NLNgsWwB6ceBYfam1YyszBrRIbGCcf1XCyJbKi2KPKVoGV2vHwq5RX3B9pWjSLLSJkpZ4lwO0NzZJpv0eoe1U0w0kN4KEmDHbssSplgoqpDWbjLUvv8mRPxqKTO9bybDgjuvxUzSNbZfLXIks62VgISJubHJCuVLAOfAHqkY4f7qv2BsLI63HNPBN9/xyYmBO9cEiBdLDtGvLxwyMColV2/PZvJur9ML195hINsmCY3F5OgXZ2mascaWJnBes4zACrXLDNLmOYX9utRh2tcY7Cnra2Nw929DLeNB0Acxt4dOF62nXYqjrVuXIAjas3PBQT3EBbp1ZJ+6/0zFJHhSpYlwqRrXb71H5QBgtSlWTI40tWuwiev2D9QWtVXVO1icGS4+hRJ9ScgQrNDr0EF6UBezQMikL8oCj0cMTloWWGK492Aj9G+eI8qf+9j6DkdDh2vF0uTBaKQyMwaSNiBfCO4XVAmxZNwtRE26FJd8Fx+1/PzDjdEOxneUx2V0AQ17snwRvT5HBKb1UknzU9ZRBN7RsloL7wkeFRo22cWpUU8/vuVs1l3yYHWR/Z9gAzsp21hsw0BP6cAeRmGGZiAHKMZ4pluzy6yBbRL7/V7RfwIH6RK8rbTzAaf+upApmT97He+qgZwgZnVDWWUueT41qWcyViyZ5w6FX1tOyGk5FcYLH6AK8o/PXBa0+os9hAZYGgs6MyOjdxTB+XPP23oMFcRpOPHbo1ixq7VJPu8mYhDXyTT4GrdqtV41yMXm1ib32EJOhIFfsNjO5U+Ami6ZW0d3gKiy5IDeTM4I6QdkHtdFmlYH6U5UY24hNR7f3yKSTOwL8aLx/7Z4B/9aEhEW1SFLt8Q0cCVUGp6K9kOAeoXQcVd0bVMfJ2HI4cio2Z/mO0TmgCeZh6esYxI+uB7T21mzvZcgDkK9CJxtb5/2ieI7dsRXLaUaB/QOTD+75IhyKc1Q1nQmLi9jOOI8Isy9NKfc0QlonWR7sqnfd9vXYYOhxQwH39VivFEvZRMsiQrax8JFoUQUh0nq3G4dBX8pzF2hiDU6Cplu7ZchpmV6wdK+FxtGMjShOFykInTr8ap3WFx25HcOtTLhThw6FM2LL/jgsLZR6o5hO9O8ps7NCGz4IUL1dh5wZvY/UJlh3ylOvf38Z2ckU7df+O0O3tgaaNzE1jIbH33yDvWhofwlORiO2OsCP4Tdcqs0BaEDBcrRG2xZHGOTEbfDn8KinFdXxmx3bZvjzsYsGtODi0y9YuL1mjn+T1fOOEYZrWbD6jMzCvYWkzY9ZOutw5STSrHYnF9+rw+0OjN2h3FkUWRObMEfAmBXjx8M6Uy8gYnfmTidfqueKlam4Df2DBL3XOvTFoOfRwH13iqic4l/gme56E/bKATrRzRcEzuf9EflrirKXh7K2kfpkLTCPTbbWtw2hDE8HjDJne1rjQkS2KOKY0fp/KbJfI1FW85WApN/xtsny2sv8R+PdG9CzFWgN+BtEa93CSanQtZ9vm/KF5Ebi758xgu9VWX0A6VqB9PzCO0i5zkf0oWewt/bAxZ5flHmeApp/z5rVu3SxoIsj8JRFS1Rawe5eqcUWJEAMpE+u0mq93eyRn3TOhy9BIeH2XYBgTQ2R1bwikUHvcScCQ2yJg13WEAVG7CsxpOQyq0ASd1zwZwcvfv3+zfMPyd9/ef/Th3fPX7xMXvzy9tXrH3mHnz4ZffPs1CYh/JehoLOC/rDpkJdIJ/5RZke2IEMIGWAemNum8wc/C7cmhgyAUE4X9X9b8IrbPlY4iU2a5ebep005X9W67Ym+nKaZryiARhd8LbXRgNQtPZIe0/l8C6iyuFY3amMAP9EHLojndPMJrJocEYiyGj4Vs7fKzlcgrKl5em3LR/HoSMqJvhJKLPXKn3ntN+WVqrxyfU+OWhiUjrUjPF1keNuKTYcFDPn2FqhGDMsksWS4ncxMS1jfdp1gMEm6aA8Xr/pa4x5D15NK5yuB+t8smI/pjxtYmADmdSMXxOGdMcvW6Y2vYk06pAnDEaD31F0x4GcHdAGZvsmDMi9hJ9QcDoy5mPrWj0Df+mFyN9uR6oSIpTCNyg2TGG8Lh9hu78bNAaaNFQgAFXUHUEsKzQqapZjJOGK0LIHKixbZytsutWKY2tE9ZlHnYHNUu8YWli5rsstOUD9OzdEzjtfH1TM7Fl89PTpmSwqUuMQLmIzuxkQbMxyYAMyF4yVGcf58QgF16NLxUqQWbMMnzwHy3Tp0sqKI+ZI3osfxIMPdFk4aEdqCKPcak/mWMbnrUXRZUmYDnaqhwxS1TJctHTD7DCsg0/wqeQwmJZ7BgiZkQdy2zYFOn+GjR5IdbvR2hTGZIIFeN+jZDPGZg7dhOsZP9ExJ7CyWho4HwkbVtkv4lF+nBbDvpL9wvl2kbo0ETVmdWmhUddhNKH+9OrhlVbGoEWRRxDNYsNU6rS58XiaCR60Sz0EM3Z6XcICu1rg8qCwDxy0SZBDahIv/5UPtJTGJxTtyCdahyCmGnoifOaxMXLQRR/whW4yEIl0mRzuBsrUoFLpVIEyCSGzULisbMlPbcDdbYbZdnFO0IXfobsKhxSfGua2BcjJYBQ4yRAM/SD0UANCX5iWA78U3nE7naYFsgW674nwL6VbCIT0eUqVXOk4SsXRCnx3LHt3A5KGunauWseo3QOGzslYyHlRcMNkDTpTaOCcdVQNe+4oG4NNWHsV6nGFSxHcyKT1xla01fAyiQ7dSphmVe76MW9LLaf/Fi7la4iqHsiwUMXt4qE/qTk+UPgL/Tk6xIV6ViA9D3XxI8Aan027znjDtHqgMp9X8K1TnMECPAlJ49XUQNd8/ROsI4nowefHmw/Rw8uHlu+nhX+rvhDji1tzDZnHpI+6Ax3XppRBxqRJrTNKmATGIjVT1hZTdRTwdMuimsPVME6I8sWFXqFdqdWAiXj10V2ZTNo2244ENizAhsfAw8AUpIMceEUpOAY620VBaLc8x6kGtDdcixbUHiAUwoSrTlus8J/2NTkFgix7b82QRcW/L9LZ86+eqwE1XVrTCyLN/1G9A795xpAiCVom0SNiex31IyN+IhNqrsroAbXA82nnL6SYDYlHrsroeO8dYVifpJezSdAYixiCyYxibXzshzkndBNotxju0z9CZxw3ygzIh9Qj9KOVy/LWhZ6DmugT0+eqSgXPOU1BGwlaPkHZIFOSibS/UZTY31iAqJPkx9K+xQrXROaAoi5yngICjmkB7L2zfZUqzjnuDO2zzUc66Gwc3F6fBJbD3kPHBdLYioZRxlL3oKOZIctiQCI3A6ixn3IcX6KyyW+y2ddWNxTfdNiWolI30lJC7kn/H+Ft7MLk2qVRHzzAABZd34daUW4GABLpp717wMjOlR494tO3MGefiYNneEtrMC8YoAPHPN1v4V+xOPuWDKMZByoAlHFgOSDetflskWnYM9dVS7s1NHP4hQotztbG80dcDkcpUYHaMHvSeKG1YGTfuB/QxdCFLNNcgpkjter9PEUTf91t70zM3AamXwd5+i7FHcnM0DNG6tI1M8cBIcTslekT2DWoGdZlfqtD3zzpV+EpOPtD1MIFDkP7RHecreP22bF5hVruRm8ylfEbXSiVIzSisfIwSWft9WcXfleylDgv48uAI1uYSZqRtw3n5MdQkjmPdyfXY6wZVN1uLRJNWlPGSofgG/5irqTVHg0LPNse5T/YupdAlT5oBmsfaCugRrn6yhP1sD3BY0QqYNAapQnPv4GZ4PAHSOStHX9Lz/buiuZZbXY3VyatAhcfZwpEMempli2M2hI8D8lV3jOEDnUb5EgkblcTjISsvdbpUzTWZFjiCD8BmM2qcX2M4X57O+ZKqNDiRNlg5dtCQGdtvbt07kzyOsTsokyaxzmqOMHTv+PjCybcy9Q5zl+MTpCOvv1rCXD90l9QeUl4kHgnBLQ3RY6hdLdBtzqejTmcQUUpD7mnsGX14Rpw9bsOYLGC2vdnbvT3ujs5z5GNYyZkbBNA3sO7hsGt0HTTc8TH4vtG1LFg6iYSDecmXM8sKenQnuS+wgJILWQsdnzhMm6+p1hBBu74jruV5npNlRAXai0K8GU/stEKJRB+thv2xWQrRlTsgDJ6Y2hGexKPgkQxpMAgePdJj98xZHqTDsf4Zr1VqArY5jsB2x0yaxcFQXvdINSfHWvobe/RsrLgGFL2Jny/S9d95N2MwHIwDvWQo8ObVWLMx1+AdeebtsT88eofXuZChWBd2LMja1kYJmXWygRKyfRIlNXAUqSx3ppcJC+dXAPYYPfUZ1ACJCVyA1u3BLAgbW+VME/cPTs4dXqHQDCYi37Hfo11ux8SPH2Bwa+mDFDigsxzpehP/CPzqA7025/PdMqrIjEBxGeW23bRjNvAO5VPxC4rMQH5uBihRrJr7UfIR+fYwh4cwcz7E4ieAoFRw6koPOHcigKOtNByREL8DcRYukDf1ZCN5l8o7e8wPNPWqtdnX4C6WyAiMeno3xh3OhWJ+bU0+XgunB68Zs0HHTsTYaJ55Bx7e7kdwmkW4UXBWZqdUKfPkQbJnthzNSScMbq+Y68fj9Gd0wBjtHfVMiM73NCi9A9Vx0ROARvkrGCN7RQQH/C8dt0RZBDdMObegFrTp4JauwIncAuESt6xFqvqQdjii1EkVQJ2T2YHNN498xkBWt45AQT25QUSGG1DkWoICB2ZAoT4Puq5yO5VdSow0L2vOQCjYsRe5dmBCINkpZ+sBIN+LrCZupUZ3OiJ/FGRd1BIra9ubchNaDXuwX1v/I/T8llLPaWV69GGIw8RrN8gEupv/W9bbLWxbLj+ulFzoTuHfqIE5bSQlnQSBVXqpyKc4LytUQJlpx70YAxXg+QX0sRORKOgeakN3xIN/AwOH+bZRy8Ahlou+BrUVUApQFYDj8dehOHcT5O2mwqAh3/ShA6e0QDH2eF7XJovsUoxT9RI9MirEnvuGIFo49gME+A7lfmN1AI475OaMeNoIK7ihP7d6S93givlXz9lTO6Y/1D8wQEsAA3JcweG/CFvt3J2PLgP8SzsgRNJFsuDZaHeHLOLxeHcF9GLjDvnTbvInQTgQjoEPHSLsc3LwOLcF/Uis2NNzl4RZeQ4BmufZhjhhglmrSa9wiR886rmUQj7hklCnFL1GWKBcwTM+2IkpDm4vlnpEfAvIoNdl0+0PIxB9tHY4TIwMyZj0wH/gWeGuLizakVnBP2EQJS3qad8UtjLnbs4O5CA8ZUJHyYFInCU/PETkhWqJXEIdFP4H5IqFFZq0Qo+YD4lIo+ApVVR5usHIef6UjtPGOceA5zln3OC2L5uP9kx5xZGa3QFoBAWrPnzw00/3xSEKjt1ziGOtPPXaqtg9Etxp54NcieGifWZ6C8WY6n0IjADfo8ERnF3l3PbSZjXl1cSNYp2az3H5r1ukthYjZH9l+WiYfDFs701PDiTR6sZ9ShddliSWYzGRA+bOWPplTd/YbITKvRfbA9gdJAZLqtFtjYK5FuZkuWYvBxH/GyNoiQOirzLMZWdXY89+tu7NewIe7Lzcf0cDemECUBNPlo9aSuC9gNM3yWjdDJTeHelGTfcSUKa/BWgVv9suu+MqO75Idy9C3j0UnZWjx8HPPZy62ND6uG2d/SdiTLG5llTK/m0oVxK3A7j3bWTtjrnvZyidxH46Lkd79yXX8sMV7lKRMSmT231nNrGJIGxd+lSp9MIzRkzQlNCQa1cYMd8n2GXFhgm3228AZnK+2WIkUUl6dHI+8+C4gUefxY9ra4doNcKbrB89OomCk7vtEaMv03PRji7M3TEQGZkgEslC3OFzTGrD29Adh9MOrE7dO/LMUCnpL5mnAD/0nYbyhR3jH+x8moaS9BcZokR3V7gmfHlwbfv+/U7Gx9IJwnMADOxdUEnnKijp3TWTR5KZ66RkNx56xmmtDRwsFegIUp3chBFR8ooYEt86X24ra+Ta7fiTzEMelX+fj1hJ3BFaEaS/Ll/6o8dKtONc8MPJiDS8Tktn1CKQ8Ah6vJbM0/jZTI3Qp03+pNZ9LlnbBvbZorwqcP55aznhk9L+PqGT0uMuuBN3xaZ8h7i3aHfHnpGPyqWDVj48A9xi5ovxqTqXdetUjXu5zIkqe/3jdnPpbxLZPeV+jUg3xWwbguyl6HRr6iW2JUAoXed75xbrnbtrz261H8bcIy/8S27n+7uD3Zn4cqfsl7kx/+W+UWQhSbHtzP2Pnd7c/S6pF3Zp2F8rCdc+pbfzvW2s5f+yu7/fO9rrF93jMNyrDXlxZvdyHLa0JvupZP9jzpJDTHvvtPu9LorI7ISKmxaDezM951PNkovmdDHpfIJYX7loDLF9yXJiLOt+93qqU2Y2dBWQUNk/sw3fWmCtangDuWQLa2SZZUxsNl/gXgWAHlYB+2UxNgKq50K+1s0AfXl0PTI3XyrnSduepO18D77/a2p7UWgxSwts4mtUNEkgO9h3jtTQ09Z4fPgCOHZRIRDXG9TTTohDvrNtHD375FE7qMSCMhKqfbXrViVbY//VSka6/ePlV+cGFnPMmk8W2nPW/VhhV1zddea6rfwz1/loX9+ZiyFoHqtof63Zagb3YxvmaPZTOQDMxDptp44TbX/3HSX9IVhojxd/dn1ilPvp/eX4tmhuOJ77LVMWwe2blvDt6KGuLUFwctT9SH8CMxFzkpX/dQbVvyydO15T38vKK+S+m/J16I18TO4VOZZYJNEfD6QMTiOROl8VRD+EtRjRR8jz3P8ACPKPOHivP23g8DyxpgAnL4vzGu9poy8Rms9syfFKfhNMmVtu85y++SqHuomnvYdAzDeC7xCHne3blYh3Wsw6W1U+IMkI1E76aCJqDaZQU3SDr/s9OOk0EhgmkwWhYC4uv7fZAgzdSdiQGpbuTuSNFU7w7de6ni+3YNGR7qXHTQnFT6TUC7mha2diIInRsRS74TZSeqSLnMgTKXrmt5KUOiqSEpTlqXvdQX/kHN2IQBH+XEtbgmj+5J2jCRBE86G6nnVlhNTi/1f3D1ndp/8Hq9uodN1ezm2RUQRRc81T/uvLN2ay5yAzbmt+/+HHD+Y9wuG3b7bz37K/lmlx/oMyxUAImzy9pjAjqXa2PVLq6Tz4DX/MZ8FfS7w9U41ASIYfR0cjfD9bGhDIsvG7oEmT0ecUBLMEsEjcPpMXR2hPaY3zc75O5GOhCZLifhLe+23TP5xgn+wm2K/30uvX//bcqEFl5P7kKq87AYmG9Hmdb/8H9HXpRw=='
embedded = json.loads(zlib.decompress(base64.b64decode(payload)))
assert all(p.startswith(('protector/', 'configs/')) for p in embedded)
for relative, contents in embedded.items():
    path = (WORK / relative).resolve()
    assert path.is_relative_to(WORK.resolve())
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents, encoding='utf-8')
sys.path.insert(0, str(WORK))
print('Embedded source files:', sorted(embedded))

In [ ]:
from protector.constants import APPROVED, ENCODERS
from protector.data import read_data, sha256, audit_data, validate_submission
from protector.training import train_experiment, finalize
assert MODEL_ID in ENCODERS
assert all(json.loads((WORK/p).read_text())['model_id'] == MODEL_ID for p in CONFIG_PATHS)
input_root = Path('/kaggle/input')
if DATA_DIR is None:
    roots = [p.parent for p in input_root.rglob('train.csv') if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists()]
    assert len(roots) == 1, f'Set DATA_DIR explicitly; candidates: {roots}'
    DATA_DIR = roots[0]
DATA_DIR = Path(DATA_DIR)
train, test, sample = read_data(DATA_DIR)
audit_data(DATA_DIR, WORK/'artifacts/audit')
if MODEL_DIR is None:
    slug = APPROVED[MODEL_ID].split('/')[1]
    candidates = [p.parent for p in input_root.rglob('config.json') if slug in str(p.parent).lower()]
    assert len(candidates) == 1, f'Attach the approved Kaggle model or set MODEL_DIR. Candidates: {candidates}'
    MODEL_DIR = candidates[0]
MODEL_DIR = Path(MODEL_DIR)
assert (MODEL_DIR/'config.json').exists()
print('Data:', DATA_DIR, 'Model:', MODEL_DIR, 'Rows:', len(train), len(test))

In [ ]:
runs = []
for config_path in CONFIG_PATHS:
    output = WORK/'artifacts'/Path(config_path).stem
    if (output/'history.json').exists():
        meta = json.loads((output/'run.json').read_text())
        assert 'total_seconds' in meta, 'Incomplete earlier run: use a fresh output directory.'
        assert meta['provenance']['train_file_sha256'] == sha256(DATA_DIR/'train.csv')
        assert meta['provenance']['split_sha256'] == sha256(WORK/'artifacts/audit/split.csv')
        assert meta['provenance']['config_file_sha256'] == sha256(WORK/config_path)
    else:
        train_experiment(DATA_DIR, WORK/'artifacts/audit/split.csv', WORK/config_path, MODEL_DIR, output)
    metric = json.loads((output/'metrics.json').read_text())
    runs.append((metric['macro_f1'], output))
    print(output.name, metric)
selected_score, selected_run = max(runs, key=lambda item: item[0])
print('Selected:', selected_run, 'validation macro F1:', selected_score)
print('Confirm small differences on another seed/split before making strong claims.')

In [ ]:
final_dir = WORK/'artifacts/final'
if not (final_dir/'submission_validation.json').exists():
    finalize(DATA_DIR, selected_run, MODEL_DIR, final_dir)
else:
    final_meta = json.loads((final_dir/'run.json').read_text())
    assert final_meta['provenance']['selected_experiment'] == str(selected_run.resolve()), 'Existing final output used another experiment; use a fresh output directory.'
validation = validate_submission(final_dir/'submission.csv', DATA_DIR)
print(json.dumps(validation, indent=2))
from IPython.display import display, FileLink
display(FileLink(str(final_dir/'submission.csv')))
print('No Kaggle submission was made. Review the CSV before authorizing submission.')

Validation scores belong to the selection model trained on the training partition. The final model is retrained on all 14,000 rows for the selected epoch count; its hidden-test score is unknown. CSV validity does not establish hidden-test performance. Class order: BENIGN, JAILBREAK, PROMPT_INJECTION. Prompt strings are always untrusted data. CUDA attention can cause small numerical differences even with fixed seeds.